# 07 · Advanced retrieval: multi-query, HyDE, HNSW, agentic loop
Inspects `ragtraining.retrieve.expand`, `ragtraining.eval.advanced` and `ragtraining.agent.loop` (LOGBOOK §7.5).
Haiku generations are **cached** (`data/expansions.json`) and HNSW is local, so this notebook costs **$0**. Needs `ollama serve`.

In [ ]:
import sys, json, pickle; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
from ragtraining.retrieve.search import Retriever
from ragtraining.retrieve.expand import alt_queries, hypothetical_passage
r = Retriever(); courses = sorted(r.chunks.course.unique())
adv = pd.read_csv('../data/eval_advanced.csv'); senses = pd.read_csv('../data/eval_ambiguous_senses.csv')

## 1. Multi-query: one question, several searches
Haiku rewrites the question 3 ways: other wording, the terms slides use, and **one query per meaning** if the term is ambiguous.

In [ ]:
q = 'What is a key?'
for x in [q] + alt_queries(q, courses): print('•', x)

## 2. How you fuse the rankings decides what you get
- **RRF** sums 1/(60+rank) over the queries → it rewards chunks *several* queries agree on, i.e. the **majority meaning**.
- **Round-robin** takes each query's next-best chunk in turn → **every meaning gets slots**.

In [ ]:
def courses_top8(**cfg):
    return r.search(q, k=8, **cfg).course.value_counts().to_dict()
for name, cfg in [('plain hybrid', dict(method='hybrid', how='rrf')), ('multi-query, RRF fuse', dict(method='multi_query')),
                  ('multi-query, round-robin', dict(method='multi_query', fuse='rr')),
                  ('multi-query RR + local rerank', dict(method='multi_query', fuse='rr', rerank='local'))]:
    print(f'{name:32s}', courses_top8(**cfg))

## 3. Sense coverage on the 4 ambiguous questions (meanings found in the top-8 chunks)

In [ ]:
senses.assign(score=senses.covered.astype(str) + '/' + senses.of.astype(str)).pivot(index='config', columns='question', values='score')

## 4. HyDE: search with a fake answer instead of the question
The passage may contain wrong details. That doesn't matter, because it's only used to *find* real pages. It works when
questions and answers are worded very differently. It can hurt when the fake answer drifts off topic.

In [ ]:
q2 = "Why can a block cipher be decrypted even when its inner scrambling function can't be inverted?"
print(hypothetical_passage(q2)[:500], '\n')
for name, cfg in [('question → cosine', dict(method='cosine')), ('HyDE passage → cosine', dict(method='hyde', dense_only=True))]:
    h = r.search(q2, k=3, **cfg)
    print(name); [print('   ', x.course[:20], '·', x.path.split('/')[-1][:45], '·', x.page_start) for x in h.itertuples()]

## 5. Retrieval quality on the 60 synthetic questions

In [ ]:
adv[['method', 'hit1', 'hit5', 'mrr', 'ndcg', 'ms_per_query']].sort_values('mrr', ascending=False)

⚠️ The synthetic questions are specific and share wording with their page, so plain hybrid is already near the ceiling
(Hit@5 0.95 with rerank). Expansion is meant for **vague and ambiguous** questions, so Max's own questions are the real test.
Expansion also costs one Haiku call (~1–2 s) per question.

## 6. HNSW vs. exact search
**HNSW** builds a graph of near neighbours and walks it instead of comparing against every vector. At 10,943 chunks exact
search already takes < 1 ms, so it isn't needed here. At millions of chunks it would be.

FAISS runs in a **separate process** (`ragtraining/eval/hnsw.py`): PyTorch (loaded above by the reranker) and FAISS each
bundle their own OpenMP runtime, and both in one process crashes the kernel on macOS (P-48).
Timing is **one query at a time**, like the tutor. Timing a batch of 200 makes exact search look ~20× faster (P-49).

In [ ]:
import subprocess
res = json.loads(subprocess.run([sys.executable, '-m', 'ragtraining.eval.hnsw'], cwd='..', capture_output=True,
                                text=True, check=True).stdout)
print(f"exact: {res['exact_ms_per_query']} ms/query (batched: {res['exact_ms_per_query_batched']})")
pd.DataFrame(res['hnsw'])

## 7. The agentic loop (`ragtraining/agent/loop.py`)
**PLAN** (Haiku: 1–3 queries + course filter) → **SEARCH** (hybrid + local rerank) → **GRADE** (Haiku: is the evidence
enough? if not, what's missing + new queries) → up to 2 more rounds, widening the course filter → **ANSWER** (Opus tutor).
The loop lives in our code, not in a hosted agent, because the data and the retriever are local.
In session 1 it rescued a vague question ("growing/shrinking phase" → two-phase locking). On RAGAS it isn't yet better
than one search (§7.3). Tuning is open: the grader's stopping rule (P-31), and multi-query inside the plan step.